# ATC Classification x WGCNA Module Eigengene Analysis

## Objective
- Analyze ATC classification (Level 1-3) x WGCNA module activity (Module Eigengene; ME)
- Visualize and quantify ME score distribution bias by ATC class

## Output
- Main figure: ATC Level 1 x ME boxplot
- Supplementary figures: ATC Level 2/3 x ME boxplot
- Statistics table: n, median, IQR, Cliff's delta, p-value, FDR

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import os
import logging
import warnings
warnings.filterwarnings('ignore')

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Project root
project_root = Path('.').resolve().parent
results_dir = project_root / 'results' / 'atc_me_analysis'
results_dir.mkdir(parents=True, exist_ok=True)

# Figure style
plt.rcParams.update({
    'font.family': 'Arial',
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'font.size': 16,
    'svg.fonttype': 'none',
    'axes.linewidth': 1.5,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f'Project root: {project_root}')
print(f'Results dir: {results_dir}')

## 1. Snowflake Connection and Data Retrieval

In [ ]:
# Snowflake connection settings
from snowflake.connector import connect
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization

def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH", "~/.ssh/snowflake_rsa_key.pem")')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    # Encode in DER format (format expected by Snowflake connector)
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user=os.environ.get("SNOWFLAKE_USER"),
            account=os.environ.get("SNOWFLAKE_ACCOUNT"),
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

# Connect
conn = connect_to_snowflake()

In [ ]:
# Get compound to InChIKey mapping
query_mapping = """
SELECT DISTINCT
    "pertname",
    "inchi_key"
FROM GLYCO_GENES_WIDE
WHERE "inchi_key" IS NOT NULL
"""

df_compound_inchi = pd.read_sql(query_mapping, conn)
print(f'Compound to InChIKey mapping retrieved: {len(df_compound_inchi)} records')
print(f'Unique pertname: {df_compound_inchi["pertname"].nunique()}')
print(f'Unique inchi_key: {df_compound_inchi["inchi_key"].nunique()}')

# Remove duplicates (1 compound 1 InChIKey)
df_compound_inchi = df_compound_inchi.drop_duplicates(subset='pertname', keep='first')
print(f'\nAfter deduplication: {len(df_compound_inchi)} compounds')

In [ ]:
# Close connection
conn.close()
print('Snowflake connection closed')

# Save
df_compound_inchi.to_csv(results_dir / 'compound_inchikey_mapping.csv', index=False)
print(f'Saved: {results_dir / "compound_inchikey_mapping.csv"}')

## 2. Loading ME Scores and ATC Data

In [ ]:
# MEスコア読み込み
df_me = pd.read_csv(project_root / 'results' / 'wgcna_v2' / 'module_eigengenes.csv', index_col=0)
print(f'MEスコア: {df_me.shape} (化合物 × モジュール)')
print(f'モジュール: {list(df_me.columns)}')

# 遺伝子モジュール読み込み
df_modules = pd.read_csv(project_root / 'results' / 'wgcna_v2' / 'gene_modules.csv')
print(f'\n遺伝子モジュール: {len(df_modules)} genes, {df_modules["module"].nunique()} modules')

In [ ]:
# ATC compound map 読み込み
df_atc = pd.read_parquet(project_root / 'results' / 'atc_master' / 'atc_compound_map.parquet')
print(f'ATC compound map: {len(df_atc)} records')
print(f'Columns: {list(df_atc.columns)}')

# ATC classification 読み込み（説明文用）
df_atc_class = pd.read_parquet(project_root / 'results' / 'atc_master' / 'atc_classification.parquet')
print(f'\nATC classification: {len(df_atc_class)} records')

## 3. Data Integration (ME x ATC)

In [ ]:
# Step 1: pertname → inchi_key
me_compounds = df_me.index.tolist()
print(f'MEスコア化合物数: {len(me_compounds)}')

# compound_inchi マッピングを辞書化
compound_to_inchi = dict(zip(df_compound_inchi['pertname'], df_compound_inchi['inchi_key']))
print(f'InChIKeyマッピング: {len(compound_to_inchi)} compounds')

# MEスコアにInChIKeyを付与
df_me_with_inchi = df_me.copy()
df_me_with_inchi['inchi_key'] = df_me_with_inchi.index.map(compound_to_inchi)

print(f'\nInChIKey付与: {df_me_with_inchi["inchi_key"].notna().sum()} / {len(df_me_with_inchi)} ({df_me_with_inchi["inchi_key"].notna().mean()*100:.1f}%)')

In [ ]:
# Step 2: inchi_key → ATC（先頭14文字で部分マッチ）
# ATCマップのInChIKey先頭14文字を抽出
df_atc['inchi_key_14'] = df_atc['INCHI_KEY'].str[:14]

# 重複除去（1 InChIKey に 1 ATC）
df_atc_unique = df_atc.drop_duplicates(subset='inchi_key_14', keep='first')
print(f'ATC unique (by inchi_key_14): {len(df_atc_unique)}')

# InChIKey先頭14文字でマッピング
df_me_with_inchi['inchi_key_14'] = df_me_with_inchi['inchi_key'].str[:14]

# ATCとマージ
df_merged = df_me_with_inchi.merge(
    df_atc_unique[['inchi_key_14', 'ATC_L1', 'ATC_L2', 'ATC_L3', 'PREF_NAME']],
    on='inchi_key_14',
    how='left'
)

print(f'\nATCマッピング成功: {df_merged["ATC_L1"].notna().sum()} / {len(df_merged)} ({df_merged["ATC_L1"].notna().mean()*100:.1f}%)')

In [ ]:
# ATC欠損の扱い: 除外
df_analysis = df_merged[df_merged['ATC_L1'].notna()].copy()
print(f'解析対象化合物数: {len(df_analysis)}')

# compound名をカラムに追加
df_analysis['compound'] = df_analysis.index
df_analysis = df_analysis.reset_index(drop=True)

# ATC Level 1 分布
print('\n=== ATC Level 1 分布 ===')
atc_l1_counts = df_analysis['ATC_L1'].value_counts().sort_values(ascending=False)
print(atc_l1_counts)

In [ ]:
# n < 20 のATCクラスを除外
MIN_SAMPLES = 20

valid_atc_l1 = atc_l1_counts[atc_l1_counts >= MIN_SAMPLES].index.tolist()
excluded_atc_l1 = atc_l1_counts[atc_l1_counts < MIN_SAMPLES].index.tolist()

print(f'閾値: n >= {MIN_SAMPLES}')
print(f'有効ATCクラス: {len(valid_atc_l1)} ({valid_atc_l1})')
print(f'除外ATCクラス: {len(excluded_atc_l1)} ({excluded_atc_l1})')

df_analysis_l1 = df_analysis[df_analysis['ATC_L1'].isin(valid_atc_l1)].copy()
print(f'\n解析対象化合物数（L1フィルタ後）: {len(df_analysis_l1)}')

In [ ]:
# ATC Level 1 の説明を追加
atc_l1_desc = {
    'A': 'Alimentary tract and metabolism',
    'B': 'Blood and blood forming organs',
    'C': 'Cardiovascular system',
    'D': 'Dermatologicals',
    'G': 'Genito-urinary system and sex hormones',
    'H': 'Systemic hormonal preparations',
    'J': 'Antiinfectives for systemic use',
    'L': 'Antineoplastic and immunomodulating agents',
    'M': 'Musculo-skeletal system',
    'N': 'Nervous system',
    'P': 'Antiparasitic products, insecticides and repellents',
    'R': 'Respiratory system',
    'S': 'Sensory organs',
    'V': 'Various'
}

# 保存: 統合データ
df_analysis.to_csv(results_dir / 'me_atc_merged.csv', index=False)
print(f'保存: {results_dir / "me_atc_merged.csv"}')

## 4. Visualization: ATC Level 1 x ME Boxplot (Main Figure)

In [ ]:
# MEカラム取得
me_cols = [c for c in df_analysis_l1.columns if c.startswith('ME')]
print(f'MEカラム数: {len(me_cols)}')

# 主要モジュール選択（分散説明率上位5つ、または手動指定）
# ここでは全モジュールから分散の大きい上位5つを選択
me_var = df_analysis_l1[me_cols].var().sort_values(ascending=False)
top_modules = me_var.head(5).index.tolist()
print(f'主要モジュール（分散上位5）: {top_modules}')

In [ ]:
# Boxplot作成: ATC Level 1 × 主要5モジュール
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=False)

# ATCクラスの並び順: サンプル数降順
atc_order = df_analysis_l1['ATC_L1'].value_counts().index.tolist()

for idx, me_col in enumerate(top_modules):
    ax = axes[idx]
    
    # Boxplot
    sns.boxplot(
        data=df_analysis_l1,
        x='ATC_L1',
        y=me_col,
        order=atc_order,
        ax=ax,
        palette='Set2',
        fliersize=2
    )
    
    ax.set_title(me_col, fontweight='bold')
    ax.set_xlabel('ATC Level 1')
    ax.set_ylabel('Module Eigengene' if idx == 0 else '')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # X軸ラベルにnを追加
    xlabels = [f'{atc}\n(n={atc_l1_counts[atc]})' for atc in atc_order]
    ax.set_xticklabels(xlabels, fontsize=16, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(results_dir / 'fig_atc_l1_me_boxplot.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'fig_atc_l1_me_boxplot.svg', format='svg', bbox_inches='tight')
plt.show()
print(f'保存: {results_dir / "fig_atc_l1_me_boxplot.svg"}')

## 5. Effect Size Calculation (Cliff's delta)

In [ ]:
def cliffs_delta(x, y):
    """
    Cliff's delta effect size.
    -1 to 1: 正は x > y、負は x < y
    |d| < 0.147: negligible
    |d| < 0.33: small
    |d| < 0.474: medium
    |d| >= 0.474: large
    """
    x = np.array(x)
    y = np.array(y)
    n1, n2 = len(x), len(y)
    
    if n1 == 0 or n2 == 0:
        return np.nan
    
    # 全ペア比較
    more = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    
    delta = (more - less) / (n1 * n2)
    return delta

def interpret_cliffs_delta(d):
    """Cliff's deltaの解釈"""
    d = abs(d)
    if d < 0.147:
        return 'negligible'
    elif d < 0.33:
        return 'small'
    elif d < 0.474:
        return 'medium'
    else:
        return 'large'

print('Cliff\'s delta関数定義完了')

In [ ]:
# 効果量計算: 各モジュール × 各ATCクラス vs 全体
results = []

for me_col in me_cols:
    # 全体のMEスコア
    all_scores = df_analysis_l1[me_col].dropna().values
    
    for atc in valid_atc_l1:
        # ATCクラスのMEスコア
        atc_scores = df_analysis_l1[df_analysis_l1['ATC_L1'] == atc][me_col].dropna().values
        # 全体からATCクラスを除いたもの
        other_scores = df_analysis_l1[df_analysis_l1['ATC_L1'] != atc][me_col].dropna().values
        
        n = len(atc_scores)
        median = np.median(atc_scores)
        q1 = np.percentile(atc_scores, 25)
        q3 = np.percentile(atc_scores, 75)
        iqr = q3 - q1
        
        # Cliff's delta（ATCクラス vs 他のすべて）
        delta = cliffs_delta(atc_scores, other_scores)
        effect_size = interpret_cliffs_delta(delta)
        
        # Wilcoxon rank-sum test
        if len(atc_scores) >= 5 and len(other_scores) >= 5:
            stat, p_value = stats.mannwhitneyu(atc_scores, other_scores, alternative='two-sided')
        else:
            p_value = np.nan
        
        results.append({
            'module': me_col,
            'atc_l1': atc,
            'atc_description': atc_l1_desc.get(atc, ''),
            'n': n,
            'median': median,
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'cliffs_delta': delta,
            'effect_size': effect_size,
            'p_value': p_value
        })

df_results = pd.DataFrame(results)
print(f'結果: {len(df_results)} rows')

In [ ]:
# 多重検定補正（BH法）
# p値が有効な行のみ
valid_p = df_results['p_value'].notna()
_, fdr, _, _ = multipletests(df_results.loc[valid_p, 'p_value'], method='fdr_bh')
df_results.loc[valid_p, 'fdr'] = fdr

# 保存
df_results.to_csv(results_dir / 'table_atc_me_statistics.csv', index=False)
print(f'保存: {results_dir / "table_atc_me_statistics.csv"}')

# 有意な結果
sig_results = df_results[(df_results['fdr'] < 0.05) & (df_results['effect_size'].isin(['medium', 'large']))]
print(f'\n有意な結果 (FDR < 0.05, medium/large effect): {len(sig_results)}')
if len(sig_results) > 0:
    print(sig_results[['module', 'atc_l1', 'n', 'cliffs_delta', 'effect_size', 'fdr']].head(20))

## 6. Supplementary Figure: ATC Level 2 x ME Boxplot

In [ ]:
# ATC Level 2 分布
atc_l2_counts = df_analysis['ATC_L2'].value_counts()
valid_atc_l2 = atc_l2_counts[atc_l2_counts >= MIN_SAMPLES].index.tolist()
print(f'ATC Level 2 有効クラス (n >= {MIN_SAMPLES}): {len(valid_atc_l2)}')
print(atc_l2_counts[atc_l2_counts >= MIN_SAMPLES])

In [ ]:
# ATC Level 2 Boxplot（上位10クラス）
top_k = 10
top_atc_l2 = atc_l2_counts.head(top_k).index.tolist()
df_analysis_l2 = df_analysis[df_analysis['ATC_L2'].isin(top_atc_l2)].copy()

fig, axes = plt.subplots(1, 5, figsize=(24, 6), sharey=False)

for idx, me_col in enumerate(top_modules):
    ax = axes[idx]
    
    sns.boxplot(
        data=df_analysis_l2,
        x='ATC_L2',
        y=me_col,
        order=top_atc_l2,
        ax=ax,
        palette='Set3',
        fliersize=2
    )
    
    ax.set_title(me_col, fontweight='bold')
    ax.set_xlabel('ATC Level 2')
    ax.set_ylabel('Module Eigengene' if idx == 0 else '')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    xlabels = [f'{atc}\n(n={atc_l2_counts[atc]})' for atc in top_atc_l2]
    ax.set_xticklabels(xlabels, fontsize=16, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(results_dir / 'fig_atc_l2_me_boxplot.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'fig_atc_l2_me_boxplot.svg', format='svg', bbox_inches='tight')
plt.show()
print(f'保存: {results_dir / "fig_atc_l2_me_boxplot.svg"}')

## 7. Supplementary Figure: ATC Level 3 x ME Boxplot

In [ ]:
# ATC Level 3 分布
atc_l3_counts = df_analysis['ATC_L3'].value_counts()
valid_atc_l3 = atc_l3_counts[atc_l3_counts >= MIN_SAMPLES].index.tolist()
print(f'ATC Level 3 有効クラス (n >= {MIN_SAMPLES}): {len(valid_atc_l3)}')
print(atc_l3_counts[atc_l3_counts >= MIN_SAMPLES].head(15))

In [ ]:
# ATC Level 3 Boxplot（上位12クラス）
top_k = 12
top_atc_l3 = atc_l3_counts.head(top_k).index.tolist()
df_analysis_l3 = df_analysis[df_analysis['ATC_L3'].isin(top_atc_l3)].copy()

fig, axes = plt.subplots(1, 5, figsize=(28, 6), sharey=False)

for idx, me_col in enumerate(top_modules):
    ax = axes[idx]
    
    sns.boxplot(
        data=df_analysis_l3,
        x='ATC_L3',
        y=me_col,
        order=top_atc_l3,
        ax=ax,
        palette='Pastel1',
        fliersize=2
    )
    
    ax.set_title(me_col, fontweight='bold')
    ax.set_xlabel('ATC Level 3')
    ax.set_ylabel('Module Eigengene' if idx == 0 else '')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    xlabels = [f'{atc}\n(n={atc_l3_counts[atc]})' for atc in top_atc_l3]
    ax.set_xticklabels(xlabels, fontsize=16, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(results_dir / 'fig_atc_l3_me_boxplot.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'fig_atc_l3_me_boxplot.svg', format='svg', bbox_inches='tight')
plt.show()
print(f'保存: {results_dir / "fig_atc_l3_me_boxplot.svg"}')

## 8. Summary

In [ ]:
print('=' * 80)
print('【解析サマリー】')
print('=' * 80)
print(f'\n入力データ:')
print(f'  MEスコア: {df_me.shape[0]} 化合物 × {df_me.shape[1]} モジュール')
print(f'  ATCマッピング成功: {len(df_analysis)} 化合物')
print(f'\n解析対象 (ATC L1, n >= {MIN_SAMPLES}):')
print(f'  有効ATCクラス: {len(valid_atc_l1)}')
print(f'  化合物数: {len(df_analysis_l1)}')
print(f'\n出力ファイル:')
for f in results_dir.glob('*'):
    print(f'  {f.name}')
print('=' * 80)